# Notebook to Compare Heuristics

In [ ]:
import asyncio
import nest_asyncio

import random
import pandas as pd
from time import time

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

In [ ]:
random.seed(42)  # For accurate comparison
nest_asyncio.apply()

In [ ]:
from src import (
    welfare_greedy,
    c_fim,
)

from src import (
    estimate_cascade_influence,
    estimate_cascade_by_community,
)

In [ ]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../../data/synthetic/networks')
path_to_results = Path('../../results/barbasi_albert/size_200/test/')

file_name = 'barbasi_albert_200'

In [ ]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')

    communities = list(greedy_modularity_communities(graph))
    costs = nx.get_node_attributes(graph, 'node_costs')

    return graph, communities, costs


graph, communities, costs = asyncio.run(main())

## Comparison of Welfare Greedy and Budgeted Welfare Greedy (CFIM)

In [ ]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
budget = 5.0  # Total budget available
num_sims = 1000

In [ ]:
results = []

for alpha in alphas:
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}

    start = time()
    cfim_seeds = c_fim(
        graph=graph,
        communities=communities,
        max_seeds=k,
        budget=budget,
        costs=costs,
        alpha=alpha,
        probability=p,
    )
    cfim_time = time() - start

    cfim_influence = estimate_cascade_influence(
        graph=graph,
        seeds=cfim_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    cfim_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=cfim_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    cfim_by_comm_rounded = {k: round(v, 2) for k, v in cfim_by_comm.items()}

    results.append(
        {
            'alpha': alpha,
            'cfim_seeds': cfim_seeds,
            'cfim_time_s': cfim_time,
            'cfim_influence_by_community': cfim_by_comm_rounded,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'org_total_influence': cfim_influence,
            'fair_total_influence': welfare_influence,
        }
    )

df = pd.DataFrame(results)
df.to_csv(f'{path_to_results}/welfare_cfim_{file_name}_{k}_results.csv', index=False)